# Clase 10 · Notebook 02
# Deep Q-Network (DQN): una red que aproxima la tabla Q

**Introducción al Deep Learning — Módulo 4, Unidad 2: Aprendizaje por refuerzo**

Cuando hay demasiados estados, la tabla Q no cabe en memoria. La solución es una **red de neuronas** que,
dado un estado, predice el valor Q de cada acción. Es una **Deep Q-Network (DQN)**.

Usaremos el **mismo gridworld** del Notebook 01 para comparar de forma justa.

1. El entorno (idéntico).
2. La red que sustituye a la tabla.
3. Entrenamiento con *experience replay*.
4. La política aprendida por la red.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. El entorno (mismo gridworld)

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from collections import deque
import random
tf.random.set_seed(42); np.random.seed(42); random.seed(42)

FILAS, COLS = 4, 4
META = (3, 3); FUEGO = [(1, 1), (2, 3)]
MOV = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}

def paso(estado, accion):
    df, dc = MOV[accion]
    nf, nc = estado[0] + df, estado[1] + dc
    if not (0 <= nf < FILAS and 0 <= nc < COLS): nf, nc = estado
    nuevo = (nf, nc)
    if nuevo == META:  return nuevo, 10, True
    if nuevo in FUEGO: return nuevo, -10, True
    return nuevo, -1, False

def estado_vector(estado):
    "Representa el estado como un vector one-hot de 16 posiciones."
    v = np.zeros(FILAS * COLS, dtype="float32")
    v[estado[0] * COLS + estado[1]] = 1.0
    return v

print("Entorno listo. Estados:", FILAS*COLS, "| Acciones: 4")

## 2. La red que sustituye a la tabla Q

Entrada: el estado (vector one-hot de 16). Salida: 4 valores Q (uno por acción). Es un MLP sencillo.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

modelo = Sequential([
    Input(shape=(16,)),
    Dense(32, activation="relu"),
    Dense(32, activation="relu"),
    Dense(4, activation="linear"),   # un valor Q por acción
])
modelo.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss="mse")
modelo.summary()

## 3. Entrenamiento con *experience replay*

Guardamos las experiencias (s, a, r, s') en una memoria y entrenamos con lotes aleatorios. Esto estabiliza
el aprendizaje. Usamos ε-greedy para explorar.

In [ ]:
memoria = deque(maxlen=2000)
gamma = 0.9
epsilon, eps_min, eps_decay = 1.0, 0.05, 0.97
EPISODIOS = 120

def entrenar_lote(batch=32):
    if len(memoria) < batch: return
    muestras = random.sample(memoria, batch)
    estados = np.array([m[0] for m in muestras])
    siguientes = np.array([m[3] for m in muestras])
    q = modelo.predict(estados, verbose=0)
    q_sig = modelo.predict(siguientes, verbose=0)
    for i, (_, a, r, _, fin) in enumerate(muestras):
        q[i][a] = r if fin else r + gamma * np.max(q_sig[i])
    modelo.fit(estados, q, epochs=1, verbose=0)

for ep in range(EPISODIOS):
    estado, fin = (0, 0), False
    for _ in range(50):
        sv = estado_vector(estado)
        if np.random.rand() < epsilon:
            a = np.random.randint(4)
        else:
            a = int(np.argmax(modelo.predict(sv[None], verbose=0)[0]))
        nuevo, r, fin = paso(estado, a)
        memoria.append((sv, a, r, estado_vector(nuevo), fin))
        estado = nuevo
        if fin: break
    entrenar_lote()
    epsilon = max(eps_min, epsilon * eps_decay)
print("Entrenamiento DQN terminado.")

## 4. La política aprendida por la red

Preguntamos a la red la mejor acción en cada casilla (sin explorar) y dibujamos la política.

In [ ]:
flechas = {0: "↑", 1: "↓", 2: "←", 3: "→"}
politica = np.full((FILAS, COLS), "", dtype=object)
for f in range(FILAS):
    for c in range(COLS):
        if (f, c) == META:    politica[f, c] = "M"
        elif (f, c) in FUEGO: politica[f, c] = "F"
        else:
            q = modelo.predict(estado_vector((f, c))[None], verbose=0)[0]
            politica[f, c] = flechas[int(np.argmax(q))]
print("Política aprendida por la DQN:\n")
for fila in politica:
    print("  ".join(fila))

La red ha aprendido una política parecida a la de la tabla Q, pero **sin guardar una tabla**: generaliza
a partir del estado. En entornos enormes (millones de estados) esta es la única opción viable.

## Experimenta tú

1. Aumenta `EPISODIOS` para mejorar la política.
2. Cambia el tamaño de la red o el `gamma`.
3. Añade más casillas de fuego o agranda la rejilla.

## Resumen

- Una **DQN** sustituye la tabla Q por una **red** que predice los valores Q de cada acción.
- **Experience replay**: entrenar con lotes aleatorios de experiencias pasadas estabiliza el aprendizaje.
- Ventaja: **generaliza** y funciona donde la tabla sería gigantesca (p. ej. los píxeles de un videojuego).

Con esto cerramos la unidad de aprendizaje por refuerzo. En la próxima clase veremos el **despliegue de modelos**.